In [1]:
import sys
sys.path.append("../")

import pandas as pd
from src.data.load import load_raw_data  # we'll adapt path manually below

# Load the processed dataset (NOT raw)
df = pd.read_csv("../data/processed/final_features.csv")

print("Shape:", df.shape)
df.head()

Shape: (10000, 33)


,year,state,district,season,crop_type,seed_variety,area_sown_hectares,irrigation_type,rainfall_mm,temperature_min_c,...,yield_tonnes_per_hectare,sowing_year,sowing_month,sowing_day,sowing_day_of_year,temperature_range_c,rainfall_per_day,total_npk_kg_ha,np_ratio,kn_ratio
0,2022,Karnataka,Belagavi,Rabi,Maize,HQPM-1,1.52,Rainfed,573.7,17.5,...,2.82,2022,11,10,314,19.5,5.794949,221.0,1.541985,1.086634
1,2019,Karnataka,Kalaburagi,Rabi,Maize,HQPM-1,22.19,Rainfed,805.9,17.3,...,2.33,2019,11,25,329,14.2,7.393578,300.3,2.497600,0.523382
2,2019,Punjab,Amritsar,Rabi,Wheat,HD-2967,8.50,Drip Irrigated,343.2,7.5,...,2.31,2019,11,15,319,18.7,2.908475,139.8,1.144231,0.475630
3,2017,Madhya Pradesh,Jabalpur,Rabi,Wheat,HD-3086,7.48,Irrigated,339.7,14.9,...,1.72,2017,10,29,302,12.3,2.342759,227.1,1.935323,0.429306
4,2022,Rajasthan,Bikaner,Kharif,Cotton,Bunny-Bt,6.44,Rainfed,564.1,20.3,...,1.46,2022,6,4,155,13.5,3.016578,200.6,5.274900,0.325529


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   year                      10000 non-null  int64  
 1   state                     10000 non-null  str    
 2   district                  10000 non-null  str    
 3   season                    10000 non-null  str    
 4   crop_type                 10000 non-null  str    
 5   seed_variety              10000 non-null  str    
 6   area_sown_hectares        10000 non-null  float64
 7   irrigation_type           10000 non-null  str    
 8   rainfall_mm               10000 non-null  float64
 9   temperature_min_c         10000 non-null  float64
 10  temperature_max_c         10000 non-null  float64
 11  temperature_avg_c         10000 non-null  float64
 12  humidity_pct              10000 non-null  float64
 13  growing_season_days       10000 non-null  int64  
 14  soil_ph           

In [3]:
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
 Series([], dtype: int64)

Duplicate rows: 0


In [4]:
print("All columns:\n", df.columns.tolist())

All columns:
 ['year', 'state', 'district', 'season', 'crop_type', 'seed_variety', 'area_sown_hectares', 'irrigation_type', 'rainfall_mm', 'temperature_min_c', 'temperature_max_c', 'temperature_avg_c', 'humidity_pct', 'growing_season_days', 'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha', 'soil_type', 'soil_moisture_pct', 'previous_yield_tonnes_ha', 'yield_trend_pct_yoy', 'ndvi', 'yield_tonnes_per_hectare', 'sowing_year', 'sowing_month', 'sowing_day', 'sowing_day_of_year', 'temperature_range_c', 'rainfall_per_day', 'total_npk_kg_ha', 'np_ratio', 'kn_ratio']


In [5]:
target_col = "yield_tonnes_per_hectare"

# Columns that should NEVER be used as features (identifiers, leakage risks)
drop_cols = ["record_id"]  # adjust once we see real columns

# Remaining feature columns
feature_cols = [c for c in df.columns if c not in drop_cols + [target_col]]

numerical_cols = df[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()

print("Target:", target_col)
print("\nDrop columns:", drop_cols)
print("\nNumerical features:", numerical_cols)
print("\nCategorical features:", categorical_cols)

Target: yield_tonnes_per_hectare

Drop columns: ['record_id']

Numerical features: ['year', 'area_sown_hectares', 'rainfall_mm', 'temperature_min_c', 'temperature_max_c', 'temperature_avg_c', 'humidity_pct', 'growing_season_days', 'soil_ph', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha', 'soil_moisture_pct', 'previous_yield_tonnes_ha', 'yield_trend_pct_yoy', 'ndvi', 'sowing_year', 'sowing_month', 'sowing_day', 'sowing_day_of_year', 'temperature_range_c', 'rainfall_per_day', 'total_npk_kg_ha', 'np_ratio', 'kn_ratio']

Categorical features: ['state', 'district', 'season', 'crop_type', 'seed_variety', 'irrigation_type', 'soil_type']


C:\Users\ACER\AppData\Local\Temp\ipykernel_23256\1585212123.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()


In [8]:
print("Correlation with target:")
print(df[["previous_yield_tonnes_ha", "ndvi", "yield_tonnes_per_hectare"]].corr()["yield_tonnes_per_hectare"])

Correlation with target:
previous_yield_tonnes_ha    0.773783
ndvi                        0.193981
yield_tonnes_per_hectare    1.000000
Name: yield_tonnes_per_hectare, dtype: float64


In [9]:
# Final feature list (adjust after leakage decision on ndvi)
X = df[feature_cols]
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

X shape: (10000, 32)
y shape: (10000,)


,year,state,district,season,crop_type,seed_variety,area_sown_hectares,irrigation_type,rainfall_mm,temperature_min_c,...,ndvi,sowing_year,sowing_month,sowing_day,sowing_day_of_year,temperature_range_c,rainfall_per_day,total_npk_kg_ha,np_ratio,kn_ratio
0,2022,Karnataka,Belagavi,Rabi,Maize,HQPM-1,1.52,Rainfed,573.7,17.5,...,0.703,2022,11,10,314,19.5,5.794949,221.0,1.541985,1.086634
1,2019,Karnataka,Kalaburagi,Rabi,Maize,HQPM-1,22.19,Rainfed,805.9,17.3,...,0.707,2019,11,25,329,14.2,7.393578,300.3,2.497600,0.523382
2,2019,Punjab,Amritsar,Rabi,Wheat,HD-2967,8.50,Drip Irrigated,343.2,7.5,...,0.649,2019,11,15,319,18.7,2.908475,139.8,1.144231,0.475630
3,2017,Madhya Pradesh,Jabalpur,Rabi,Wheat,HD-3086,7.48,Irrigated,339.7,14.9,...,0.575,2017,10,29,302,12.3,2.342759,227.1,1.935323,0.429306
4,2022,Rajasthan,Bikaner,Kharif,Cotton,Bunny-Bt,6.44,Rainfed,564.1,20.3,...,0.651,2022,6,4,155,13.5,3.016578,200.6,5.274900,0.325529


In [3]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.4 MB 10.3 MB/s eta 0:00:01
   -------------------------------------- - 8.1/8.4 MB 26.9 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 24.9 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ------- -------------------------------- 7.1/37.4 MB 33.3 MB/s eta 0:00:01
   ---------------- ----------------------- 15.7/37.4 MB 36.8 MB/s eta 0:00:01
   ------------------------- -------------- 23.9/37.4 MB 38.1 MB/s eta 0:00:01
   ------------------------------- -------- 29.6/37.4 MB 35.9 MB/s eta 0:00:01
   ------------------------------------- -- 35.1/37.4 MB 33.9 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 32.2 MB/s  0:00:01

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [s


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

ModuleNotFoundError: No module named 'sklearn'

In [8]:
!python -m venv venv